## WWF Priority Place — Amazon Binary Classifier

### 1 · Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
WWF_GREEN  = '#006B3C'
WWF_GOLD   = '#F5A623'
AMAZON_CLR = '#E8742A'
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9'})

### 2 · Keyword Config

In [ ]:
# Segment review order per Training_Rules.pdf
SEGMENT_ORDER = [
    'Cost Center',   # Level 1
    'Program Code',  # Level 2
    'Grant',         # Level 3
]

# To add a new Priority Place: add a key here with its keyword list
PLACE_KEYWORDS = {
    'Amazon': [
        # Countries (primary + additional per Training_Rules.pdf)
        'amazon', 'brazil', 'brasil', 'bolivia', 'peru', 'ecuador',
        'colombia', 'guyana', 'suriname', 'surinam', 'french guiana',
        # Landscapes & rivers
        'tapajos', 'pantanal', 'cerrado',
        # Species (English)
        'jaguar',
        # Known Amazon-specific programs/grants
        'arpa', 'heco', 'pfp', 'tfff',
    ],
    # 'Congo Basin':             ['congo', 'drc', 'gabon', 'cameroon', 'salonga', ...],
    # 'Eastern Himalayas':       ['himalaya', 'bhutan', 'nepal', 'india', ...],
    # 'Great Plains':            ['great plains', 'chihuahuan', 'bison', 'prairie', ...],
    # 'Arctic':                  ['arctic', 'alaska', 'greenland', 'polar bear', ...],
    # 'Greater Mekong':          ['mekong', 'cambodia', 'myanmar', 'annamites', ...],
    # 'Southern Africa':         ['southern africa', 'kaza', 'zambezi', 'namibia', ...],
    # 'SW Indian Ocean':         ['swio', 'madagascar', 'mozambique', 'tanzania', ...],
    # 'SW Pacific + Indonesia':  ['indonesia', 'philippines', 'papua', 'borneo', ...],
    # 'Eastern Pacific Seascape': ['galapagos', 'eastern pacific', 'cocos', 'cmar', ...],
}

### 3 · Pipeline 1 — Rule-Based Keyword Scan

In [3]:
def rule_scan_row(row, place_keywords=PLACE_KEYWORDS, segment_order=SEGMENT_ORDER):
    for segment in segment_order:
        text = str(row.get(segment) or '').lower()
        for place, keywords in place_keywords.items():
            if any(kw in text for kw in keywords):
                return place
    return None

def apply_rules(df):
    return df.apply(rule_scan_row, axis=1)

### 4 · Load Data & Build Training Set

In [4]:
raw = pd.read_csv('Model Training Data.csv', encoding='latin1')
raw.columns = raw.columns.str.strip()
raw.drop(columns=['Unnamed: 7', 'Unnamed: 8'], inplace=True, errors='ignore')

pp = raw['Priority Place'].astype(str).str.strip().str.lower()

# Positives: any row tagged as Amazon
amazon_mask = pp.str.contains('amazon', na=False)

# Negatives: rows tagged with a different, confirmed Priority Place
neg_mask = raw['Priority Place'].notna() & ~amazon_mask

# Untagged rows are excluded from training, predicted on later
raw['label'] = np.where(amazon_mask, 1, np.where(neg_mask, 0, -1))
train_df = raw[raw['label'] != -1].copy()

print(f"Amazon (positives) : {int((train_df['label']==1).sum())}")
print(f"Not Amazon (negatives): {int((train_df['label']==0).sum())}")
print(f"Untagged (excluded): {int((raw['label']==-1).sum())}")

Amazon (positives) : 52
Not Amazon (negatives): 230
Untagged (excluded): 594


### 5 · Pipeline 2 — ML Model

In [5]:
# Segment fields + Country + Big Bet as text features
TEXT_COLS = SEGMENT_ORDER + ['Country', 'Big Bet']

def make_text(row):
    parts = [str(row.get(c, '') or '') for c in TEXT_COLS]
    return ' '.join(p for p in parts if p and p.lower() not in ('nan', ''))

train_df['text'] = train_df.apply(make_text, axis=1)
X = list(train_df['text'])
y = list(train_df['label'].astype(int))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ml_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2), min_df=1, max_features=50000, sublinear_tf=True,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=42,
    )),
])
ml_pipe.fit(X_train, y_train)

THRESHOLD = 0.20
print('Model trained.')

Model trained.


### 6 · Evaluate ML Pipeline

In [ ]:
y_proba = ml_pipe.predict_proba(X_test)[:, 1]
y_pred  = (y_proba >= THRESHOLD).astype(int)

print(f'Test accuracy    : {accuracy_score(y_test, y_pred)*100:.1f}%')
print(f'Amazon precision : {precision_score(y_test, y_pred, pos_label=1)*100:.1f}%')
print(f'Amazon recall    : {recall_score(y_test, y_pred, pos_label=1)*100:.1f}%')
print(f'Amazon F1        : {f1_score(y_test, y_pred, pos_label=1):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Amazon', 'Amazon'], zero_division=0))

from sklearn.base import clone
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_arr, y_arr = np.array(X), np.array(y)
fold_accs, fold_recs, fold_pres = [], [], []
for tr_idx, te_idx in cv.split(X_arr, y_arr):
    p = clone(ml_pipe)
    p.fit(X_arr[tr_idx], y_arr[tr_idx])
    proba = p.predict_proba(X_arr[te_idx])[:, 1]
    yp = (proba >= THRESHOLD).astype(int)
    fold_accs.append(accuracy_score(y_arr[te_idx], yp))
    fold_recs.append(recall_score(y_arr[te_idx], yp, pos_label=1, zero_division=0))
    fold_pres.append(precision_score(y_arr[te_idx], yp, pos_label=1, zero_division=0))

print(f'CV scores : {[f"{a*100:.1f}%" for a in fold_accs]}')
print(f'CV mean   : {np.mean(fold_accs)*100:.1f}%  +/-  {np.std(fold_accs)*100:.1f}%')
print(f'CV recall : {np.mean(fold_recs)*100:.1f}%  +/-  {np.std(fold_recs)*100:.1f}%')
print(f'Target    : 80.0%  ->  {"PASS!" if np.mean(fold_accs) >= 0.80 else "FAIL :("}')

Test accuracy    : 93.0%
Amazon precision : 81.8%
Amazon recall    : 81.8%
Amazon F1        : 0.8182

              precision    recall  f1-score   support

  Not Amazon       0.96      0.96      0.96        46
      Amazon       0.82      0.82      0.82        11

    accuracy                           0.93        57
   macro avg       0.89      0.89      0.89        57
weighted avg       0.93      0.93      0.93        57

CV scores : ['93.0%', '93.0%', '87.5%', '89.3%', '94.6%']
CV mean   : 91.5%  +/-  2.7%
CV recall : 88.2%  +/-  11.7%
Target    : 80.0%  ->  PASS!


### 7 · Apply Both Pipelines to All Rows

In [ ]:
raw['rule_tag'] = apply_rules(raw)
raw['text']     = raw.apply(make_text, axis=1)
raw['ml_proba'] = ml_pipe.predict_proba(raw['text'].tolist())[:, 1]
raw['ml_tag']   = (raw['ml_proba'] >= THRESHOLD).astype(int).map({1: 'Amazon', 0: None})

raw['final_tag'] = raw.apply(
    lambda r: r['rule_tag'] if r['rule_tag'] is not None else r['ml_tag'], axis=1
)

n_rule = int(((raw['rule_tag'].notna()) & (raw['ml_tag'].isna())).sum())
n_ml   = int(((raw['rule_tag'].isna())  & (raw['ml_tag'].notna())).sum())
n_both = int(((raw['rule_tag'].notna()) & (raw['ml_tag'].notna())).sum())
n_tot  = int(raw['final_tag'].notna().sum())

print(f'Total rows tagged Amazon : {n_tot}')
print(f'  Rule only              : {n_rule}')
print(f'  ML only                : {n_ml}')
print(f'  Both agreed            : {n_both}')

labeled = raw[raw['label'] != -1].copy()
y_true  = labeled['label'].astype(int).tolist()
y_comb  = (labeled['final_tag'] == 'Amazon').astype(int).tolist()
print(f'\nCombined accuracy on labeled rows : {accuracy_score(y_true, y_comb)*100:.1f}%')
print(f'Combined recall                   : {recall_score(y_true, y_comb, pos_label=1)*100:.1f}%')
print(f'Combined precision                : {precision_score(y_true, y_comb, pos_label=1)*100:.1f}%')

Total rows tagged Amazon : 168
  Rule only              : 9
  ML only                : 79
  Both agreed            : 80

Combined accuracy on labeled rows : 97.9%
Combined recall                   : 98.1%
Combined precision                : 91.1%
